# 03 - Error Analysis

This notebook loads a trained checkpoint, evaluates it on the validation set,
and performs detailed error analysis including:
- Confusion matrix visualization
- Per-class accuracy breakdown
- Most commonly confused sign pairs (hard negatives)
- Top-1 and Top-5 accuracy summary

**Prerequisites:**
- A trained model checkpoint in `checkpoints/best_model.pt`
- Preprocessed keypoint data in `data/processed/`
- Split CSVs in `data/splits/`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Model and Data

Load the trained model from checkpoint and build the validation DataLoader.

In [ ]:
from src.training.config import load_config
from src.models.pose_transformer import build_pose_model
from src.data.dataset import WLASLKeypointDataset, get_dataloader
from src.data.augment import get_val_transforms
from src.training.evaluate import compute_metrics, plot_confusion_matrix, find_hard_negatives

# Configuration
config_path = PROJECT_ROOT / "configs" / "pose_transformer.yaml"
checkpoint_path = PROJECT_ROOT / "checkpoints" / "best_model.pt"

cfg = load_config(config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Approach: {cfg.approach}")
print(f"Num classes: {cfg.num_classes}")

In [ ]:
# Build model and load weights
model = build_pose_model(cfg).to(device)

if checkpoint_path.exists():
    ckpt = torch.load(str(checkpoint_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')}")
    print(f"Best val top-1: {ckpt.get('best_top1', '?')}%")
else:
    print("WARNING: No checkpoint found. Using randomly initialized model.")
    print("Train a model first with: python -m src.training.train --config configs/pose_transformer.yaml")

model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Build validation dataset
data_dir = PROJECT_ROOT / "data"
val_csv = data_dir / "splits" / f"WLASL{cfg.wlasl_variant}" / "val.csv"

if val_csv.exists():
    val_transform = get_val_transforms(T=cfg.T)
    val_dataset = WLASLKeypointDataset(
        split_csv=val_csv,
        keypoint_dir=data_dir / "processed",
        transform=val_transform,
        T=cfg.T,
    )
    val_loader = get_dataloader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    print(f"Validation set: {len(val_dataset)} samples")
    
    # Build class names
    val_df = pd.read_csv(val_csv)
    class_names = [""] * cfg.num_classes
    for _, row in val_df.iterrows():
        idx = int(row["label_idx"])
        if idx < cfg.num_classes:
            class_names[idx] = row["gloss"]
else:
    print(f"No validation split found at {val_csv}. Run preprocessing first.")
    val_loader = None
    class_names = [str(i) for i in range(cfg.num_classes)]

## 2. Compute Metrics

Run the model on the entire validation set and compute top-1/top-5
accuracy, per-class accuracy, and the confusion matrix.

In [ ]:
if val_loader is not None and len(val_dataset) > 0:
    metrics = compute_metrics(model, val_loader, device, class_names, approach=cfg.approach)
    
    print(f"Top-1 Accuracy: {metrics['top1']:.2f}%")
    print(f"Top-5 Accuracy: {metrics['top5']:.2f}%")
else:
    print("Cannot compute metrics without validation data.")
    metrics = None

## 3. Confusion Matrix

The confusion matrix reveals which signs are most commonly misclassified.
Diagonal entries represent correct predictions.

In [ ]:
if metrics is not None:
    cm = metrics["confusion_matrix"]
    
    # Save full confusion matrix
    output_dir = PROJECT_ROOT / "eval_results"
    output_dir.mkdir(exist_ok=True)
    plot_confusion_matrix(cm, class_names, output_dir / "confusion_matrix.png")
    
    # Also display inline (smaller version if many classes)
    n = len(class_names)
    if n <= 30:
        fig_size = max(8, n * 0.4)
        fig, ax = plt.subplots(figsize=(fig_size, fig_size))
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm = np.divide(cm.astype(float), row_sums, 
                           out=np.zeros_like(cm, dtype=float), where=row_sums > 0)
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title("Normalized Confusion Matrix")
        plt.tight_layout()
        plt.show()
    else:
        print(f"Confusion matrix saved to {output_dir / 'confusion_matrix.png'}")
        print(f"(Too many classes ({n}) to display inline; see saved file.)")

## 4. Per-Class Accuracy

Identify which classes the model handles well and which it struggles with.

In [ ]:
if metrics is not None:
    per_class = metrics["per_class_accuracy"]
    acc_df = pd.DataFrame([
        {"gloss": name, "accuracy": acc}
        for name, acc in per_class.items()
        if name  # skip empty names
    ]).sort_values("accuracy", ascending=True)
    
    print(f"Mean per-class accuracy: {acc_df['accuracy'].mean():.1f}%")
    print(f"Classes with 0% accuracy: {(acc_df['accuracy'] == 0).sum()}")
    print(f"Classes with 100% accuracy: {(acc_df['accuracy'] == 100).sum()}")
    
    # Bottom-10 and Top-10
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    bottom10 = acc_df.head(10)
    axes[0].barh(bottom10["gloss"], bottom10["accuracy"], color="coral")
    axes[0].set_xlabel("Accuracy (%)")
    axes[0].set_title("10 Hardest Classes")
    axes[0].set_xlim(0, 100)
    
    top10 = acc_df.tail(10)
    axes[1].barh(top10["gloss"], top10["accuracy"], color="seagreen")
    axes[1].set_xlabel("Accuracy (%)")
    axes[1].set_title("10 Easiest Classes")
    axes[1].set_xlim(0, 100)
    
    plt.tight_layout()
    plt.show()
    
    # Distribution of per-class accuracies
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(acc_df["accuracy"].values, bins=20, color="steelblue", edgecolor="white")
    ax.axvline(acc_df["accuracy"].mean(), color="red", linestyle="--",
               label=f"Mean: {acc_df['accuracy'].mean():.1f}%")
    ax.set_xlabel("Per-Class Accuracy (%)")
    ax.set_ylabel("Number of Classes")
    ax.set_title("Distribution of Per-Class Accuracies")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Most Confused Class Pairs

These are the sign pairs that the model most frequently confuses with each other.
Understanding these helps guide further data collection or model improvements.

In [ ]:
if metrics is not None:
    pairs = find_hard_negatives(metrics["confusion_matrix"], class_names, top_k=15)
    
    print("Most Confused Class Pairs:")
    print(f"{'True Class':<25} {'Predicted As':<25} {'Count':>6}")
    print("-" * 60)
    for true_cls, pred_cls, count in pairs:
        print(f"{true_cls:<25} {pred_cls:<25} {count:>6}")
    
    # Visualize as a bar chart
    if pairs:
        fig, ax = plt.subplots(figsize=(12, 6))
        labels = [f"{t} -> {p}" for t, p, c in pairs]
        counts = [c for t, p, c in pairs]
        ax.barh(labels[::-1], counts[::-1], color="coral")
        ax.set_xlabel("Misclassification Count")
        ax.set_title("Top Confused Class Pairs")
        plt.tight_layout()
        plt.show()

## 6. Accuracy Summary Table

A consolidated view of all key metrics.

In [ ]:
if metrics is not None:
    summary = {
        "Metric": ["Top-1 Accuracy", "Top-5 Accuracy", "Mean Per-Class Accuracy",
                   "Classes with >80% Acc", "Classes with 0% Acc"],
        "Value": [
            f"{metrics['top1']:.2f}%",
            f"{metrics['top5']:.2f}%",
            f"{acc_df['accuracy'].mean():.2f}%",
            str((acc_df['accuracy'] > 80).sum()),
            str((acc_df['accuracy'] == 0).sum()),
        ]
    }
    summary_df = pd.DataFrame(summary)
    display(summary_df.style.hide(axis="index"))
else:
    print("No metrics available. Train a model and preprocess data first.")